# Notebook 05 · Correct and freeze the reviewed pilot oracles

Use a CPU Kaggle session. Attach the exact Stage 03 and Stage 04 datasets and a dataset containing `stage_05_review_inputs_v2.json`.

Run Cell 1 to rebuild and verify the original oracle packets, apply the versioned corrections, and write `/kaggle/working/stage_05_oracle_review_v3/oracle_review_packets.zip`. Review the changed packets and complete the pending rows in its CSV template. Attach your completed CSV as `oracle_review_completed.csv`. Then run Cell 2 to publish Stage 05.

The review template carries forward approvals for packets with matching bytes. The correction audit records the original, v2, and final packet hashes, the technical changes, and the applicable counts. If adding the CSV restarts Kaggle, rerun Cell 1 with the same datasets before Cell 2.


In [3]:
import csv
import hashlib
import io
import json
import os
import re
import zipfile
from concurrent.futures import ThreadPoolExecutor
from time import perf_counter
from pathlib import Path, PurePosixPath

INPUT = Path(os.environ.get("OPENAPI_INPUT", "/kaggle/input"))
WORK = Path(os.environ.get("OPENAPI_WORK", "/kaggle/working"))
DESIGN = "openapi-fresh-pilot-v1"


def sha(data):
    return hashlib.sha256(data).hexdigest()


def file_sha(path):
    with path.open("rb") as f:
        return hashlib.file_digest(f, "sha256").hexdigest()


def encode(data):
    return (json.dumps(data, sort_keys=True, ensure_ascii=False, separators=(",", ":"),
                       allow_nan=False) + "\n").encode()


def load(path):
    return json.loads(path.read_bytes())


def records(path):
    with path.open(encoding="utf-8") as f:
        for line in f:
            if line.strip():
                yield json.loads(line)


def csv_records(path):
    with path.open(newline="", encoding="utf-8-sig") as f:
        return list(csv.DictReader(f))


def atomic(path, data):
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_bytes(data)
    tmp.replace(path)


def jsonl(path, rows):
    atomic(path, b"".join(map(encode, rows)))


def csv_write(path, fields, rows):
    output = io.StringIO(newline="")
    writer = csv.DictWriter(output, fieldnames=fields, lineterminator="\n")
    writer.writeheader()
    writer.writerows(rows)
    atomic(path, output.getvalue().encode())


def safe_path(name):
    p = PurePosixPath(name)
    if p.is_absolute() or ".." in p.parts or "\\" in name or p.as_posix() != name:
        raise ValueError(f"Unsafe manifest path: {name}")
    return Path(*p.parts)


import hashlib
import json
import re
from pathlib import PurePosixPath


ARTIFACT_SCHEMA = "openapi-artifact-v1"
PRODUCER_NOTEBOOKS = {
    "00": "00_protocol_and_runtime",
    "01": "01_corpus_preparation_and_eligibility",
    "02": "02_operation_slicing_and_prose_measurement",
    "03": "03_pair_and_evaluator_feasibility",
    "04": "04_strata_and_pilot_selection",
    "05": "05_oracle_build_and_review",
    "06": "06_build_corrected_evaluator_and_generation_tasks",
    "07": "07_pilot_generation",
    "08": "08_pilot_execution_and_review",
}


def provenance_digest(value):
    data = json.dumps(value, sort_keys=True, separators=(",", ":"),
                      ensure_ascii=False, allow_nan=False).encode("utf-8")
    return hashlib.sha256(data).hexdigest()


def artifact_path_for(record):
    paths = {record[key] for key in ("artifact_path", "relative_path", "path")
             if key in record}
    if len(paths) != 1:
        raise RuntimeError("Missing or inconsistent artifact paths")
    name = paths.pop()
    path = PurePosixPath(name)
    if not name or name == "." or path.is_absolute() or ".." in path.parts or "\\" in name or path.as_posix() != name:
        raise RuntimeError(f"Invalid artifact path: {name!r}")
    return name


def manifest_digest_valid(value):
    return isinstance(value, str) and re.fullmatch(r"[0-9a-f]{64}", value) is not None


def register_artifact_provenance(manifest):
    stage = manifest["stage"]
    artifacts = manifest["artifacts"]
    if "upstream_manifest_sha256" not in manifest:
        if stage != "00":
            raise RuntimeError(f"Stage {stage} has no upstream identity")
        manifest["upstream_manifest_sha256"] = {}
    links = manifest["upstream_manifest_sha256"]
    config_hash = manifest.get("configuration_hash")
    if config_hash is None:
        config_hash = provenance_digest({artifact_path_for(row): row["sha256"]
                                         for row in artifacts
                                         if artifact_path_for(row).startswith("config/")})
    manifest["configuration_hash"] = config_hash
    manifest["artifact_record_schema"] = ARTIFACT_SCHEMA
    upstream_hash = provenance_digest(links)
    for row in artifacts:
        row.update(artifact_path=artifact_path_for(row), schema_version=ARTIFACT_SCHEMA,
                   producer_notebook=PRODUCER_NOTEBOOKS[stage], config_hash=config_hash,
                   upstream_manifest_hash=upstream_hash)
    verify_artifact_provenance(manifest)


def verify_artifact_provenance(manifest):
    stage = manifest["stage"]
    if manifest.get("artifact_record_schema") != ARTIFACT_SCHEMA:
        raise RuntimeError(f"Stage {stage} requires a corrected provenance export")
    links = manifest["upstream_manifest_sha256"]
    config_hash = manifest["configuration_hash"]
    if (not isinstance(links, dict) or bool(links) != (stage != "00")
            or any(key not in PRODUCER_NOTEBOOKS or key >= stage or not manifest_digest_valid(value)
                   for key, value in links.items()) or not manifest_digest_valid(config_hash)):
        raise RuntimeError(f"Invalid Stage {stage} provenance")
    expected = (ARTIFACT_SCHEMA, PRODUCER_NOTEBOOKS[stage], config_hash, provenance_digest(links))
    seen = set()
    artifacts = manifest["artifacts"]
    if not artifacts or len(artifacts) != manifest["artifact_count"]:
        raise RuntimeError(f"Invalid Stage {stage} artifact count")
    for row in artifacts:
        name = artifact_path_for(row)
        actual = tuple(row.get(key) for key in
                       ("schema_version", "producer_notebook", "config_hash", "upstream_manifest_hash"))
        if (row.get("artifact_path") != name or name in seen or actual != expected
                or not manifest_digest_valid(row.get("sha256"))
                or type(row.get("bytes")) is not int or row["bytes"] < 0):
            raise RuntimeError(f"Invalid Stage {stage} artifact provenance: {name}")
        seen.add(name)


def stage_input(stage, *, fresh=True):
    started = perf_counter()
    print(f"Stage {stage}: locating the attached manifest under {INPUT}", flush=True)
    matches = []
    for external in sorted(INPUT.rglob(f"stage_{stage}_manifest.json")):
        sidecar = load(external)
        if fresh and sidecar.get("design_id") != DESIGN:
            continue
        for manifest in external.parent.rglob(f"registries/{stage}_manifest.json"):
            matches.append((external, manifest))
    if len(matches) != 1:
        raise RuntimeError(f"Attach one expanded Stage {stage} release. Found {len(matches)}.")
    external, internal = matches[0]
    sidecar, manifest = load(external), load(internal)
    expected = sidecar.get("internal_manifest_sha256") or sidecar.get("internal_manifest", {}).get("sha256")
    if expected != file_sha(internal):
        raise RuntimeError(f"Stage {stage} manifest digest mismatch.")
    if (manifest.get("stage_status", manifest.get("status")) != "PASS"
            or sidecar.get("stage_status", sidecar.get("status")) != "PASS"):
        raise RuntimeError(f"Stage {stage} has no passing handoff.")
    verify_artifact_provenance(manifest)
    root = internal.parent.parent
    def verify(item):
        name = artifact_path_for(item)
        path = root / safe_path(name)
        if path.is_symlink() or not path.is_file() or path.stat().st_size != item["bytes"]:
            raise RuntimeError(f"Stage {stage} artifact missing or wrong size: {name}")
        if file_sha(path) != item["sha256"]:
            raise RuntimeError(f"Stage {stage} artifact SHA-256 mismatch: {name}")
        return name, item["sha256"]

    artifacts = manifest["artifacts"]
    total = len(artifacts)
    workers = min(16, max(4, (os.cpu_count() or 2) * 4))
    print(f"Stage {stage}: verifying {total:,} artifacts with {workers} workers", flush=True)
    paths = {}
    batch_size = 1_000
    with ThreadPoolExecutor(max_workers=workers) as executor:
        for start in range(0, total, batch_size):
            batch = artifacts[start:start + batch_size]
            paths.update(executor.map(verify, batch))
            elapsed = perf_counter() - started
            print(f"Stage {stage}: verified {len(paths):,}/{total:,} artifacts ({elapsed:.1f}s)", flush=True)
    if len(paths) != manifest["artifact_count"]:
        raise RuntimeError(f"Stage {stage} artifact count mismatch.")
    return root, manifest, expected, paths


def pack(stage, root, metadata):
    output = WORK / f"stage_{stage}_fresh_dataset_v3"
    output.mkdir(parents=True, exist_ok=True)
    manifest_path = root / f"registries/{stage}_manifest.json"
    files = sorted(p for p in root.rglob("*") if p.is_file() and p != manifest_path
                   and "__pycache__" not in p.parts and not p.name.endswith(".tmp"))
    artifacts = [{"relative_path": p.relative_to(root).as_posix(), "bytes": p.stat().st_size,
                  "sha256": file_sha(p)} for p in files]
    manifest = {"stage": stage, "status": "PASS", "design_id": DESIGN,
                "artifact_count": len(artifacts), "artifacts": artifacts, **metadata}
    register_artifact_provenance(manifest)
    atomic(manifest_path, encode(manifest))
    archive = output / f"stage_{stage}_fresh.zip"
    tmp = archive.with_suffix(".tmp")
    with zipfile.ZipFile(tmp, "w", zipfile.ZIP_DEFLATED, compresslevel=4) as z:
        for p in [*files, manifest_path]:
            z.write(p, "prose_operation_experiment/" + p.relative_to(root).as_posix())
    tmp.replace(archive)
    external = {k: v for k, v in manifest.items() if k != "artifacts"}
    external.update(archive=archive.name, archive_sha256=file_sha(archive),
                    archive_bytes=archive.stat().st_size, internal_manifest_sha256=file_sha(manifest_path))
    atomic(output / f"stage_{stage}_manifest.json", encode(external))
    atomic(output / f"stage_{stage}_checksums.sha256",
           f'{external["archive_sha256"]}  {archive.name}\n'.encode())
    atomic(output / "README.md", f"# Stage {stage} fresh pilot\n\nDesign {DESIGN}.\n".encode())
    print(f"Stage {stage} PASS. Publish {output}")
    return external


from collections import Counter
import shutil
import sys

import jsonschema

if "date-time" not in jsonschema.FormatChecker().checkers:
    raise RuntimeError(
        "Stage 03 used JSON Schema date-time validation. Run "
        "%pip install rfc3339-validator==0.1.4, restart the kernel, "
        "and rerun this cell."
    )

stage03, manifest03, hash03, artifacts03 = stage_input("03")
stage04, manifest04, hash04, artifacts04 = stage_input("04")
if manifest04["upstream_manifest_sha256"]["03"] != hash03 or not manifest04["checkpoint_a_reached"]:
    raise RuntimeError("Stage 04 selection does not belong to the attached Stage 03 frame.")
project = WORK / "fresh_release/stage_05_v3/prose_operation_experiment"
review_root = WORK / "stage_05_oracle_review_v3"
review_root.mkdir(parents=True, exist_ok=True)
qualification = load(stage03 / "reports/audit/feasibility_builder_qualification.json")
if qualification.get("success_status_qualification", {}).get("ambiguous_statuses_blocked") != "PASS":
    raise RuntimeError("Stage 03 must qualify the shared success-status policy.")
for name, digest in qualification["module_hashes"].items():
    if artifacts03.get(f"src/{name}") != digest:
        raise RuntimeError("Shared oracle builder failed provenance verification.")
sys.path.insert(0, str(stage03 / "src"))
import oracle
if (Path(oracle.__file__).resolve() != (stage03 / "src/oracle.py").resolve()
        or oracle.ORACLE_VERSION != qualification["builder_version"]):
    raise RuntimeError("Start Notebook 05 in a clean session with the qualified Stage 03 builder.")
contracts_path = stage03 / "config/oracle_record_contracts.json"
contracts = load(contracts_path)
if artifacts03["config/oracle_record_contracts.json"] != qualification["contract_sha256"]:
    raise RuntimeError("Shared oracle format changed.")
ranked = list(records(stage04 / "partitions/pilot/ranked_frame.jsonl"))
strata = csv_records(stage04 / "partitions/pilot/stratum_sizes.csv")
allocation = {row["stratum"]: int(row["n_h"]) for row in strata}
exclusion_files = sorted(INPUT.rglob("frame_eligibility_exclusions.csv"))
if len(exclusion_files) > 1:
    raise RuntimeError("Attach at most one reviewed frame exclusion ledger.")
exclusions = csv_records(exclusion_files[0]) if exclusion_files else []
excluded = {row["op_id"] for row in exclusions}
frame_ids = {row["op_id"] for row in ranked}
if (len(excluded) != len(exclusions) or not excluded <= frame_ids
        or any(row["stage03_manifest_sha256"] != hash03
               or row["reason_code"] != "PREGENERATION_ELIGIBILITY_FAILURE"
               or not row["reviewer"].strip() or not row["evidence"].strip()
               or not row["uniform_frame_rule"].strip() for row in exclusions)):
    raise RuntimeError("Frame exclusions must be tied to a uniformly applied, reviewed eligibility rule.")
# Require a declared, machine-checkable exclusion applied to the entire ranked frame.
rules = set()
for exclusion in exclusions:
    rule = json.loads(exclusion["uniform_frame_rule"])
    if (not isinstance(rule, dict) or set(rule) != {"field", "equals"}
            or rule["field"] not in {"spec_id", "family_id"}
            or not isinstance(rule["equals"], str) or not rule["equals"]):
        raise RuntimeError("An exclusion rule must identify a proven ineligible specification or family.")
    rules.add((rule["field"], rule["equals"]))
    original = next(row for row in ranked if row["op_id"] == exclusion["op_id"])
    if original[rule["field"]] != rule["equals"]:
        raise RuntimeError("The declared eligibility rule does not exclude this operation.")
if excluded != {row["op_id"] for row in ranked if any(row[field] == value for field, value in rules)}:
    raise RuntimeError("Apply the reviewed rule to the entire frame before selecting reserve replacements.")
eligible_sizes = Counter(row["stratum"] for row in ranked if row["op_id"] not in excluded)
kept = Counter()
selection, review_rows = [], []
packets = review_root / "packets"
total_packets = sum(allocation.values())
build_started = perf_counter()
print(f"Building {total_packets} draft oracle review packets", flush=True)
for row in ranked:
    op, stratum = row["op_id"], row["stratum"]
    if op in excluded or kept[stratum] >= allocation[stratum]:
        continue
    n, N = allocation[stratum], eligible_sizes[stratum]
    if N < n:
        raise RuntimeError("Eligible stratum exhausted. Rebuild the frame and draw.")
    kept[stratum] += 1
    l0 = stage03 / f"data/slices/L0/{op}.json"
    if file_sha(l0) != row["L0_sha256"]:
        raise RuntimeError("Frozen L0 changed: " + op)
    package = oracle.build_oracle(load(l0), op, row["method"], row["path"], contracts)
    if sha(encode(package)) != row["oracle_sha256"]:
        raise RuntimeError("Oracle differs from Stage 03 feasibility: " + op)
    selection.append({**row, "L0_hash": row["L0_sha256"], "L1_hash": row["L1_sha256"],
                      "final_selection_status": "RETAINED_ORIGINAL" if row["selection_rank"] <= n
                      else "QUALIFIED_SAME_STRATUM_RESERVE",
                      "inclusion_probability": n / N, "sampling_weight": N / n,
                      "final_stratum_N": N, "stratum_population_share": N / sum(eligible_sizes.values())})
    packet = {"op_id": op, "stage04_manifest_sha256": hash04, "l0_sha256": row["L0_sha256"],
              "oracle_sha256": row["oracle_sha256"], "document": load(l0), "records": package}
    packet_path = packets / f"{op}.json"
    atomic(packet_path, encode(packet))
    review_rows.append({"op_id": op, "packet_sha256": file_sha(packet_path),
                        "traceability": "", "baseline_validity": "", "variation_validity": "",
                        "fault_validity": "", "scenario_mapping": "", "denominator_reconciliation": "",
                        "ambiguities_resolved": "", "decision": "", "reviewer": "", "evidence_notes": ""})
    completed = len(review_rows)
    if completed % 10 == 0 or completed == total_packets:
        elapsed = perf_counter() - build_started
        print(f"Built {completed}/{total_packets} review packets ({elapsed:.1f}s)", flush=True)
if kept != Counter(allocation):
    raise RuntimeError("Final selection does not fill every stratum.")
EXPECTED_REVIEW_INPUT_SHA256 = "e3daea766b8c2acbc0cfb18a644035c32c3dd159f3dede2e9a0159d729438d81"
REVIEW_VERSION = "human-review-v3"
REVIEW_CHECKLIST = (
    "traceability", "baseline_validity", "variation_validity", "fault_validity",
    "scenario_mapping", "denominator_reconciliation", "ambiguities_resolved",
)


def approval_matches(row, expected_hash):
    return (
        row["packet_sha256"] == expected_hash
        and row["decision"] == "APPROVE"
        and all(row[field] == "PASS" for field in REVIEW_CHECKLIST)
        and bool(row["reviewer"].strip())
        and bool(row["evidence_notes"].strip())
    )


def qualify_packet(packet):
    document, package = packet["document"], packet["records"]
    oracle.validate_records(contracts, package)
    scenarios = {s["scenario_id"]: s for s in package["scenarios"]}
    baselines = {b["baseline_id"]: b for b in package["baselines"]}
    mapped = {key for key, s in scenarios.items() if s.get("documented_response_mapping") is not None}
    if Counter(b["scenario_id"] for b in baselines.values()) != Counter(mapped):
        raise RuntimeError(f"{packet['op_id']}: expected one baseline per mapped scenario.")
    for baseline in baselines.values():
        scenario = scenarios[baseline["scenario_id"]]
        oracle.audit_response(document, scenario, baseline, baseline)
        request = oracle.routing.prepare_frozen_fixture(document, scenario)
        match = oracle.routing.classify_request(request, package["scenarios"], document)
        if oracle.routing.select_frozen_response(match, package["baselines"]) != baseline:
            raise RuntimeError(f"{packet['op_id']}: baseline fixture replay changed scenario.")
    applicable = {r["requirement_id"] for r in package["requirements"] if r["applicability"] == "APPLICABLE"}
    for key, fault in (("legitimate_variations", False), ("response_faults", True)):
        for record in package[key]:
            baseline = baselines[record["baseline_id"]]
            scenario = scenarios[baseline["scenario_id"]]
            if (record.get("scenario_id") not in (None, baseline["scenario_id"])
                    or (fault and (record["requirement_id"] not in applicable
                                   or record["non_equivalent"] is not True))):
                raise RuntimeError(f"{packet['op_id']}: invalid replay record linkage.")
            oracle.audit_response(document, scenario, record, baseline, fault=fault)
    counts = {
        key: sum(r["applicability"] == "APPLICABLE" for r in package[key])
        for key in ("scenarios", "requirements", "response_faults")
    }
    if not all(counts.values()):
        raise RuntimeError(f"{packet['op_id']}: zero applicable scoring denominator.")
    return counts


def correct_packet(original, revised, entry):
    package = revised["records"]
    old_scenarios = {s["scenario_id"]: s for s in original["records"]["scenarios"]}
    proposed = [
        s for s in package["scenarios"]
        if s.get("documented_response_mapping") != old_scenarios[s["scenario_id"]].get("documented_response_mapping")
    ]
    if len(proposed) != entry["mapped_invalid_scenarios"]:
        raise RuntimeError(f"{revised['op_id']}: mapping inventory differs from correction manifest.")
    mapping_decisions = [r for r in entry["decisions"] if r["action"] == "MAP_INVALID_STATUS"]
    if bool(proposed) != bool(mapping_decisions) or len(mapping_decisions) > 1:
        raise RuntimeError(f"{revised['op_id']}: expected one recorded mapping decision.")
    mapped, excluded_cases = [], []
    for scenario in proposed:
        sid = scenario["scenario_id"]
        status = scenario["documented_response_mapping"]["status"]
        decision = mapping_decisions[0]
        pointer = ("/paths/" + scenario["path_template"].replace("~", "~0").replace("/", "~1")
                   + "/" + scenario["method"] + f"/responses/{status}/description")
        response = revised["document"]["paths"][scenario["path_template"]][scenario["method"]]["responses"][str(status)]
        if (scenario["expected_validity"] != "INVALID" or old_scenarios[sid]["documented_response_mapping"] is not None
                or decision["source_pointer"] != pointer or decision["source_text"] != response.get("description")):
            raise RuntimeError(f"{sid}: invalid status proposal lacks the recorded OpenAPI evidence.")
        try:
            request = oracle.routing.prepare_frozen_fixture(revised["document"], scenario)
            match = oracle.routing.classify_request(request, package["scenarios"], revised["document"])
            reason = None if match["scenario_id"] == sid and not match["positive"] else "WIRE_REQUEST_MATCHED_DIFFERENT_SCENARIO"
        except oracle.routing.EvaluatorUnavailable as error:
            reason = error.code
        if reason:
            # A fixture without a distinguishable request cannot enter the scoring denominator.
            scenario.update(documented_response_mapping=None, applicability="NOT_APPLICABLE",
                            diagnostic=f"{REVIEW_VERSION}: mapping for HTTP {status} unavailable in offline replay ({reason}).")
            excluded_cases.append({"scenario_id": sid, "proposed_status": status, "source_pointer": pointer, "reason": reason})
            continue
        built = oracle.oracle_baselines.build_baselines(
            revised["document"], revised["op_id"], scenario["method"], scenario["path_template"], [scenario]
        )
        if built["blocking"] or len(built["baselines"]) != 1:
            raise RuntimeError(f"{sid}: a mapped scenario requires a valid response baseline.")
        baseline = built["baselines"][0]
        package["baselines"].append(baseline)
        mapped.append({"scenario_id": sid, "status": status, "source_pointer": pointer, "baseline_id": baseline["baseline_id"]})

    for req in package["requirements"]:
        if req["review_status"] == "HUMAN_VERIFIED":
            req["review_status"] = "APPROVED"
        if mapping_decisions and not mapped and req["requirement_id"] == mapping_decisions[0]["requirement_id"]:
            req.update(applicability="NOT_APPLICABLE",
                       diagnostic=f"{REVIEW_VERSION}: documented error rule has no distinguishable offline fixture.")
    return mapped, excluded_cases


def apply_review_corrections():
    sources = sorted(INPUT.rglob("stage_05_review_inputs_v2.json"))
    if len(sources) != 1:
        raise RuntimeError("Attach the dataset containing stage_05_review_inputs_v2.json.")
    source_bytes = sources[0].read_bytes()
    if sha(source_bytes) != EXPECTED_REVIEW_INPUT_SHA256:
        raise RuntimeError("Review input SHA-256 mismatch. Attach the original review input dataset.")
    source = json.loads(source_bytes)
    manifest = source["correction_manifest"]
    entries = {r["op_id"]: r for r in manifest["operations"]}
    approvals = {r["op_id"]: r for r in csv.DictReader(io.StringIO(source["approval_csv"]))}
    if set(entries) != {r["op_id"] for r in selection} or set(approvals) != set(entries) or set(source["packets"]) != set(entries):
        raise RuntimeError("The review input must cover the exact frozen pilot selection.")
    audit = []
    for row, review in zip(selection, review_rows, strict=True):
        op = row["op_id"]
        entry = entries[op]
        path = packets / f"{op}.json"
        original_bytes = path.read_bytes()
        v2_bytes = source["packets"][op].encode("utf-8")
        original_hash, v2_hash = sha(original_bytes), sha(v2_bytes)
        if (entry["source_packet_sha256"] != original_hash or entry["corrected_packet_sha256"] != v2_hash
                or entry["changed"] != (original_bytes != v2_bytes) or not approval_matches(approvals[op], v2_hash)):
            raise RuntimeError(f"{op}: v2 packet lineage or approvals do not match.")
        original, revised = json.loads(original_bytes), json.loads(v2_bytes)
        if any(original[key] != revised[key] for key in ("op_id", "document", "l0_sha256", "stage04_manifest_sha256")):
            raise RuntimeError(f"{op}: corrected packet does not match the frozen source operation.")
        old_oracle_hash = row["oracle_sha256"]
        if entry["changed"]:
            if (revised["correction_version"] != manifest["version"]
                    or revised["corrected_from_packet_sha256"] != original_hash
                    or revised["oracle_sha256"] != sha(encode(revised["records"])[:-1])):
                raise RuntimeError(f"{op}: invalid v2 record digest or version.")
            mapped, unavailable = correct_packet(original, revised, entry)
            revised.update(correction_version=REVIEW_VERSION, corrected_from_packet_sha256=original_hash,
                           oracle_sha256=sha(encode(revised["records"])))
            final_bytes = encode(revised)
        else:
            mapped, unavailable = [], []
            final_bytes = original_bytes
        counts = qualify_packet(revised)
        final_hash = sha(final_bytes)
        row.update(source_oracle_sha256=old_oracle_hash, oracle_sha256=revised["oracle_sha256"])
        atomic(path, final_bytes)
        review["packet_sha256"] = final_hash
        if approval_matches(approvals[op], final_hash):
            review.update(approvals[op])
        audit.append({
            "op_id": op,
            "original_packet_sha256": original_hash,
            "original_oracle_sha256": old_oracle_hash,
            "v2_packet_sha256": v2_hash,
            "final_packet_sha256": final_hash,
            "final_oracle_sha256": row["oracle_sha256"],
            "v2_adjudication": entry,
            "validated_new_mappings": mapped,
            "unavailable_offline_fixtures": unavailable,
            "applicable_counts": counts,
            "needs_new_approval": final_hash != v2_hash,
        })
    return {
        "version": REVIEW_VERSION,
        "review_input_sha256": sha(source_bytes),
        "stage_03_manifest_sha256": hash03,
        "stage_04_manifest_sha256": hash04,
        "v2_policy": manifest["policy"],
        "operations": audit,
    }


correction_audit = apply_review_corrections()
correction_audit_bytes = encode(correction_audit)
atomic(review_root / "oracle_corrections_v3.json", correction_audit_bytes)

csv_write(review_root / "oracle_review_template.csv", list(review_rows[0]), review_rows)
csv_write(review_root / "frame_eligibility_exclusions_template.csv",
          ["op_id", "stage03_manifest_sha256", "reason_code", "reviewer", "evidence", "uniform_frame_rule"], [])
print("Writing the oracle review ZIP", flush=True)
with zipfile.ZipFile(review_root / "oracle_review_packets.zip", "w", zipfile.ZIP_DEFLATED) as archive:
    archive.write(review_root / "oracle_review_template.csv", "oracle_review_template.csv")
    archive.write(review_root / "oracle_corrections_v3.json", "oracle_corrections_v3.json")
    for row in review_rows:
        packet = packets / f"{row['op_id']}.json"
        archive.write(packet, f"packets/{packet.name}")
pending = sum(row["decision"] != "APPROVE" for row in review_rows)
print(f"{pending} corrected packets require approval. Download {review_root / 'oracle_review_packets.zip'}.")


Stage 03: locating the attached manifest under /kaggle/input
Stage 03: verifying 36,561 artifacts with 16 workers
Stage 03: verified 1,000/36,561 artifacts (1.9s)
Stage 03: verified 2,000/36,561 artifacts (2.4s)
Stage 03: verified 3,000/36,561 artifacts (2.9s)
Stage 03: verified 4,000/36,561 artifacts (3.5s)
Stage 03: verified 5,000/36,561 artifacts (4.0s)
Stage 03: verified 6,000/36,561 artifacts (4.5s)
Stage 03: verified 7,000/36,561 artifacts (5.0s)
Stage 03: verified 8,000/36,561 artifacts (5.5s)
Stage 03: verified 9,000/36,561 artifacts (6.0s)
Stage 03: verified 10,000/36,561 artifacts (6.6s)
Stage 03: verified 11,000/36,561 artifacts (7.0s)
Stage 03: verified 12,000/36,561 artifacts (7.5s)
Stage 03: verified 13,000/36,561 artifacts (8.0s)
Stage 03: verified 14,000/36,561 artifacts (8.6s)
Stage 03: verified 15,000/36,561 artifacts (9.0s)
Stage 03: verified 16,000/36,561 artifacts (9.5s)
Stage 03: verified 17,000/36,561 artifacts (10.0s)
Stage 03: verified 18,000/36,561 artifacts (

In [4]:
decision_files = sorted(INPUT.rglob("oracle_review_completed.csv"))
if len(decision_files) != 1:
    raise RuntimeError("Attach one completed oracle_review_completed.csv after human review.")
decisions = csv_records(decision_files[0])
by_id = {row["op_id"]: row for row in decisions}
expected = {row["op_id"]: row for row in review_rows}
if len(by_id) != len(decisions) or set(by_id) != set(expected):
    raise RuntimeError("Oracle decisions must cover the full selected set once.")
bad_hashes = [op for op, row in by_id.items() if row["packet_sha256"] != expected[op]["packet_sha256"]]
if bad_hashes:
    raise RuntimeError(f"{len(bad_hashes)} approvals refer to different packets. Use the v3 review template. First: {bad_hashes[0]}")
pending = [op for op, row in by_id.items() if not approval_matches(row, expected[op]["packet_sha256"])]
if pending:
    raise RuntimeError(f"{len(pending)} packets still need approval. First: {pending[0]}")
for row in selection:
    op = row["op_id"]
    packet_path = packets / f"{op}.json"
    if file_sha(packet_path) != by_id[op]["packet_sha256"]:
        raise RuntimeError("Oracle packet changed after review: " + op)
    packet = load(packet_path)
    if sha(encode(packet["records"])) != row["oracle_sha256"]:
        raise RuntimeError("Oracle record hash does not match the qualified packet: " + op)
published_manifest = WORK / "stage_05_fresh_dataset_v3/stage_05_manifest.json"
if published_manifest.exists():
    raise RuntimeError(f"Stage 05 v3 is already published at {published_manifest.parent}.")
# Rebuild the output tree only after all approvals and their hashes pass.
if project.exists():
    shutil.rmtree(project)
for row in selection:
    op = row["op_id"]
    packet = load(packets / f"{op}.json")
    folder = project / f"oracles/frozen/pilot/{op}"
    checks = {}
    for kind, items in packet["records"].items():
        path = folder / f"{kind}.jsonl"
        jsonl(path, items)
        checks[path.name] = file_sha(path)
    atomic(folder / "validation.json", encode({
        "op_id": op, "status": "PASS", "artifact_sha256": checks,
        "oracle_sha256": row["oracle_sha256"], "review_packet_sha256": by_id[op]["packet_sha256"],
        "human_review_complete": True,
    }))
    for condition in ("L0", "L1"):
        name = f"data/slices/{condition}/{op}.json"
        data = (stage03 / name).read_bytes()
        if sha(data) != row[f"{condition}_hash"]:
            raise RuntimeError("Frozen pair changed: " + op)
        atomic(project / f"partitions/pilot/inputs/{condition}/{op}.json", data)
atomic(project / "config/oracle_record_contracts.json", contracts_path.read_bytes())
atomic(project / "reports/audit/oracle_corrections_v3.json", correction_audit_bytes)
jsonl(project / "partitions/pilot/final_selected_operations.jsonl", selection)
csv_write(project / "reports/audit/oracle_review_completed.csv", list(decisions[0]), decisions)
jsonl(project / "reports/audit/frame_eligibility_exclusions.jsonl", exclusions)
jsonl(project / "partitions/pilot/final_stratum_sizes.jsonl", [
    {"stratum": h, "N_h": N, "W_h": N / sum(eligible_sizes.values()), "n_h": allocation[h],
     "selection_probability": allocation[h] / N, "sampling_weight": N / allocation[h]}
    for h, N in sorted(eligible_sizes.items())
])
pack("05", project, {"upstream_manifest_sha256": {"00": manifest03["upstream_manifest_sha256"]["00"],
                                                 "03": hash03, "04": hash04},
                     "oracle_review_version": REVIEW_VERSION,
                     "oracle_correction_audit_sha256": sha(correction_audit_bytes),
                     "checkpoint_b_reached": True, "may_begin_notebook_06": True,
                     "human_review_complete": True, "model_generation_performed": False,
                     "operation_count": len(selection), "stratum_counts": allocation,
                     "routing_sha256": qualification["module_hashes"]["routing.py"],
                     "success_status_qualification": qualification["success_status_qualification"]})


Stage 05 PASS. Publish /kaggle/working/stage_05_fresh_dataset_v3


{'stage': '05',
 'status': 'PASS',
 'design_id': 'openapi-fresh-pilot-v1',
 'artifact_count': 906,
 'upstream_manifest_sha256': {'00': 'a064f3bb9814b3545ed81984c5a5bcd5a5ce236ce621f2fd0d039950efafc431',
  '03': '74550c1645772a46e668cbbc4024fb36728cff14a27781694e851483251a9272',
  '04': '74efa268b726abf8badbc408d909608c59ab1bda84f5e2037f50cfd7b0692e8d'},
 'oracle_review_version': 'human-review-v3',
 'oracle_correction_audit_sha256': 'ad170488f58aaceb965197c2cbdafd23cada875c59ea29e3e7cebb3da51ce615',
 'checkpoint_b_reached': True,
 'may_begin_notebook_06': True,
 'human_review_complete': True,
 'model_generation_performed': False,
 'operation_count': 100,
 'stratum_counts': {'low': 34, 'medium': 33, 'high': 33},
 'routing_sha256': '79bce0046c73935d9eaeebf57987535db7bfc05a78490d9105db1df3d6be3898',
 'success_status_qualification': {'ambiguous_statuses_blocked': 'PASS',
  'default_only_unmapped': 'PASS',
  'policy': 'single-explicit-success-v1',
  'unique_explicit_status': 'PASS'},
 'confi